In [1]:
import requests
import litellm
import urllib.parse

In [16]:
def wikipedia_tool(topic):
    # exact URL structure requested
    url = f"https://en.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&exintro&explaintext&redirects=1&titles={urllib.parse.quote(topic)}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
    }
    response = requests.get(url, headers=headers).json()
    pages = response.get("query", {}).get("pages", {})
    # Extract the text from the first page returned
    page_data = next(iter(pages.values()), {})
    return page_data.get("extract", "No information found.")

In [17]:
wikipedia_tool("Python (programming language)")

'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming.\nGuido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project\'s annual release cycle and five-year support policy. Python 3.15 is currently in the alpha development phase, and the stable release is expected to come out in October 2026."Earlier versions

In [18]:
def calculator_tool(expression):
    # exact URL structure requested
    url = f"https://api.mathjs.org/v4/?expr={urllib.parse.quote(expression)}"
    response = requests.get(url)
    return response.text if response.status_code == 200 else "Error in calculation"

In [19]:
calculator_tool("2+2")

'4'

In [43]:
SYSTEM_PROMPT = """You are a helpful assistant with access to tools. 
To solve a task, you MUST follow this loop:

Thought: Reason about what you need to do.
Action: Call a tool in this format: wikipedia[topic] OR calculator[expression]
Observation: (Wait for the system to provide the result)
Answer: Provide the final response.

Available Tools:
- wikipedia[topic]: Gets a summary of a topic.
- calculator[expression]: Evaluates a math expression. Without commas and in standard notation.

=== Examples ===
User: What is the capital of France?
Thought: I need to find out the capital of France.
Action: wikipedia[France]
Observation: Paris is the capital of France.
Answer: The capital of France is Paris.

User: What is the square of the population of Vatican City?
Thought: I need to find the population of Vatican City first.
Action: wikipedia[Vatican City]
Observation: Vatican City has a population of about 800.
Thought: Now I need to calculate the square of 800.
Action: calculator[800^2]
Observation: 640000
Answer: The square of the population of Vatican City is 640000.

You must end with "Answer: <final answer>" after getting the necessary observations.
"""

In [53]:
def run_react(user_prompt):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, 
                {"role": "user", "content": user_prompt}]
    
    for i in range(10):  # Limit steps to prevent infinite loops
        # LLM generates Thought and Action
        response = litellm.completion(
            model="openai/gpt-4o",
            messages=messages,
            stop=["Observation:"]  # Crucial: we provide the observation, not the LLM
        )
        llm_output = response.choices[0].message.content
        print(f"\n[LLM]:\n{llm_output}")
        messages.append({"role": "assistant", "content": llm_output})

        if "Answer:" in llm_output:
            break

        # Parse and Execute Action
        if "wikipedia[" in llm_output:
            topic = llm_output.split("wikipedia[")[1].split("]")[0]
            observation = wikipedia_tool(topic)
        elif "calculator[" in llm_output:
            expr = llm_output.split("calculator[")[1].split("]")[0]
            observation = calculator_tool(expr)
        else:
            observation = None

        if observation:
            # Feed Observation back into context (The "Restart" phase)
            print(f"[Action]:\nObservation: {observation[:100]}...") # truncate for display
            messages.append({"role": "user", "content": f"Observation: {observation}"})

    return messages

In [54]:
msgs = run_react("What is a age of universe in seconds?")


[LLM]:
Thought: I first need to find the age of the universe in years, then convert that into seconds.
Action: wikipedia[Age of the universe]

[Action]:
Observation: In Big Bang models of physical cosmology, the age of the universe is the cosmological time back to t...

[LLM]:
Thought: The age of the universe is approximately 13.79 billion years. Now I need to convert 13.79 billion years into seconds.

1 year = 365.25 days (including leap year correction)
1 day = 24 hours
1 hour = 3600 seconds

So, 13.79 billion years in seconds is calculated as \(13.79 \times 10^9 \times 365.25 \times 24 \times 3600\).

Action: calculator[13.79 * 10^9 * 365.25 * 24 * 3600]

[Action]:
Observation: 4.35179304e+17...

[LLM]:
Answer: The age of the universe is approximately \(4.35 \times 10^{17}\) seconds.


In [55]:
msgs

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools. \nTo solve a task, you MUST follow this loop:\n\nThought: Reason about what you need to do.\nAction: Call a tool in this format: wikipedia[topic] OR calculator[expression]\nObservation: (Wait for the system to provide the result)\nAnswer: Provide the final response.\n\nAvailable Tools:\n- wikipedia[topic]: Gets a summary of a topic.\n- calculator[expression]: Evaluates a math expression. Without commas and in standard notation.\n\n=== Examples ===\nUser: What is the capital of France?\nThought: I need to find out the capital of France.\nAction: wikipedia[France]\nObservation: Paris is the capital of France.\nAnswer: The capital of France is Paris.\n\nUser: What is the square of the population of Vatican City?\nThought: I need to find the population of Vatican City first.\nAction: wikipedia[Vatican City]\nObservation: Vatican City has a population of about 800.\nThought: Now I need to calculate the squar

In [56]:
msgs = run_react("What is the ratio of volume of Earth to volume of Moon?")


[LLM]:
To find the ratio of the volume of Earth to the volume of the Moon, I need to get their respective volumes first.

Action: wikipedia[Earth]

[Action]:
Observation: Earth is the third planet from the Sun and the only astronomical object known to harbor life. This i...

[LLM]:
It seems I did not get the required information about Earth's volume. Let me refine my search for the specific data on Earth's volume.

Action: wikipedia[Earth volume]
[Action]:
Observation: Earth radius (denoted as R🜨 or RE) is the distance from the center of Earth to a point on or near it...

[LLM]:
I received information on Earth's radius but not directly on its volume. To find the volume of Earth, we can use the formula for the volume of a sphere: \( V = \frac{4}{3} \pi r^3 \), assuming Earth is approximately spherical with an average radius of 6,371 km.

Next, I need to find the volume of the Moon.

Action: wikipedia[Moon]
[Action]:
Observation: The Moon is the only natural satellite of Earth. It orbit

In [57]:
msgs

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools. \nTo solve a task, you MUST follow this loop:\n\nThought: Reason about what you need to do.\nAction: Call a tool in this format: wikipedia[topic] OR calculator[expression]\nObservation: (Wait for the system to provide the result)\nAnswer: Provide the final response.\n\nAvailable Tools:\n- wikipedia[topic]: Gets a summary of a topic.\n- calculator[expression]: Evaluates a math expression. Without commas and in standard notation.\n\n=== Examples ===\nUser: What is the capital of France?\nThought: I need to find out the capital of France.\nAction: wikipedia[France]\nObservation: Paris is the capital of France.\nAnswer: The capital of France is Paris.\n\nUser: What is the square of the population of Vatican City?\nThought: I need to find the population of Vatican City first.\nAction: wikipedia[Vatican City]\nObservation: Vatican City has a population of about 800.\nThought: Now I need to calculate the squar